# CDL row crop vs GLAD GLC cropland: agreement, lost pixels, and emissions impact

This notebook quantifies the disagreement between the USDA Cropland Data Layer (CDL) and GLAD GLCLUC v2 cropland classifications, and assesses whether that disagreement creates a bias in the jdLUC methodology's emissions factors.

**Why this matters.** The methodology uses GLAD GLC's cropland mask (pixel value 244) as the canonical 2020 cropland extent. CDL is then layered on top to attribute pixels to specific row crops (corn, soy, wheat). Any CDL row-crop pixel that GLAD GLC does not also classify as cropland is excluded from both the emissions numerator and the production denominator. We need to confirm that the excluded pixels do not introduce a systematic bias in the per-crop emissions factors.

**Result preview.** ~91% of CONUS CDL row crop hectares are confirmed by GLAD GLC. The remaining 9% are concentrated in fragmented agricultural regions (Southeast, Northeast). Decomposing the disagreement on 10 large agricultural states shows that nearly all of the lost area is stable built-up or stable short vegetation — most of which would not have produced LUC emissions even if classified as cropland. Only ~3% of lost pixels show a transition history that would have generated meaningful emissions if counted.

The findings here support the [GLAD GLC vs CDL row crop comparison](../docs/cdl_glad_comparison_supplement.md) supplement.

## Setup

Earth Engine is used for all raster math. Results are cached to `analyses/output/` as JSON Lines so the notebook can be re-run without re-querying GEE.

In [ ]:
import json
import logging
import os
from pathlib import Path
from typing import Any

import ee
import pandas as pd

from jdluc.utils.constants import (
    GCP_PROJECT,
    GEE_CDL_COLLECTION,
    GEE_TIGER_STATES,
)
from jdluc.utils.gee import initialize_gee

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)

# GCP_PROJECT comes from utils.constants (env-var overridable via JDLUC_GCP_PROJECT).
initialize_gee(GCP_PROJECT)

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

EEImage = Any
EEGeometry = Any

### Constants

**GLAD GLCLUC** v2 uses pixel value `244` for cropland; we also reclassify the full value range into nine simplified categories for source/destination breakdowns.

**CDL row crop codes** are the herbaceous crop codes (annual + perennial) excluding hay (codes 36, 37) and fallow (61), since the wider category churns year-to-year and over-counts vs GLC. Section 4 documents the empirical basis for that exclusion.

In [ ]:
GEE_GLAD_GLC_PREFIX = 'projects/glad/GLCLU2020/v2/LCLUC_'
GLAD_GLC_CROPLAND_VALUE = 244

# CDL herbaceous crop codes (annual + perennial herbaceous, incl. hay/fallow).
CDL_HERBACEOUS_CROP_CODES = [
    1, 2, 3, 4, 5, 6, 10, 11, 12, 13, 14,
    21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
    36, 37, 38, 39,
    41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 56, 57,
    61,
    205, 206, 207, 208, 209, 213, 214, 216, 218, 219, 221, 222, 223,
    225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237,
    238, 239, 240, 241, 243, 244, 245, 246, 247, 248, 249, 254,
]
CDL_HAY_CODES = [36, 37]
CDL_FALLOW_CODES = [61]
CDL_ROWCROP_CODES = [
    c for c in CDL_HERBACEOUS_CROP_CODES
    if c not in CDL_HAY_CODES and c not in CDL_FALLOW_CODES
]

# GLAD GLC simplified categories
GLAD_CATEGORY_NAMES = {
    0: 'bare', 1: 'short_veg', 2: 'forest',
    3: 'wetland_short_veg', 4: 'wetland_forest',
    5: 'water', 6: 'cropland', 7: 'built_up', 8: 'snow_ice_other',
}

# CONUS: exclude AK (02), HI (15), and territories
EXCLUDED_STATEFP = {'02', '15', '60', '66', '69', '72', '78'}

# Two states are too large for one reduceRegion; tile them.
LARGE_STATES = {'Texas', 'New Mexico', 'Montana'}

# Anchor analysis on CDL 2020 × GLAD GLC 2020 (same-year baseline).
GLAD_YEAR = 2020
CDL_YEAR = 2020

# Subset of large agricultural states for the deeper Case A/B/C analyses.
STATES_FOR_DEEP_DIVE = [
    'Iowa', 'North Dakota', 'Illinois', 'Montana', 'Georgia',
    'Kansas', 'Texas', 'Minnesota', 'Indiana', 'Nebraska',
]

### Helper functions

Image loaders, geometry tiling for large states, and a small caching utility that keeps each state's result on its own line so an interrupted run resumes cleanly.

In [ ]:
def load_glad_glc_crop_mask(year: int, geometry: EEGeometry) -> EEImage:
    return (
        ee.Image(GEE_GLAD_GLC_PREFIX + str(year))
        .clip(geometry)
        .eq(GLAD_GLC_CROPLAND_VALUE)
        .rename('glad_glc_crop')
    )


def load_cdl_rowcrop_mask(year: int, geometry: EEGeometry) -> EEImage:
    cdl = (
        ee.ImageCollection(GEE_CDL_COLLECTION)
        .filter(ee.Filter.calendarRange(year, year, 'year'))
        .first()
        .select('cropland')
        .clip(geometry)
    )
    return cdl.remap(
        CDL_ROWCROP_CODES, [1] * len(CDL_ROWCROP_CODES), defaultValue=0
    ).rename('cdl_rowcrop')


def classify_glad_glc(image: EEImage) -> EEImage:
    """Reclassify GLAD GLC raw values to the nine simplified categories."""
    raw, cat = [0], [0]
    for v in range(1, 25):  raw.append(v); cat.append(1)  # short veg
    for v in range(25, 49): raw.append(v); cat.append(2)  # forest
    for v in range(100, 125): raw.append(v); cat.append(3)  # wetland short
    for v in range(125, 149): raw.append(v); cat.append(4)  # wetland forest
    for v in range(200, 208): raw.append(v); cat.append(5)  # water
    raw += [244, 250, 241]; cat += [6, 7, 8]                # crop, built, snow
    return image.remap(raw, cat, defaultValue=8)


def tile_geometry(geometry: EEGeometry, n_cols: int = 3, n_rows: int = 3) -> list[EEGeometry]:
    bounds = ee.Geometry(geometry).bounds().getInfo()['coordinates'][0]
    lons = [p[0] for p in bounds]; lats = [p[1] for p in bounds]
    min_lon, max_lon = min(lons), max(lons)
    min_lat, max_lat = min(lats), max(lats)
    d_lon = (max_lon - min_lon) / n_cols
    d_lat = (max_lat - min_lat) / n_rows
    geom_ee = ee.Geometry(geometry)
    tiles = []
    for r in range(n_rows):
        for c in range(n_cols):
            tile_rect = ee.Geometry.Rectangle([
                min_lon + c * d_lon, min_lat + r * d_lat,
                min_lon + (c + 1) * d_lon, min_lat + (r + 1) * d_lat,
            ])
            tiles.append(geom_ee.intersection(tile_rect))
    return tiles


def get_conus_states() -> dict[str, EEGeometry]:
    states = ee.FeatureCollection(GEE_TIGER_STATES).filter(
        ee.Filter.inList('STATEFP', list(EXCLUDED_STATEFP)).Not()
    )
    feats = states.getInfo()['features']
    return {f['properties']['NAME']: f['geometry'] for f in feats}


def cached_per_state(
    cache_path: Path,
    states: list[str],
    state_geoms: dict[str, EEGeometry],
    compute_fn,
) -> list[dict[str, Any]]:
    """Run compute_fn(state, geometry) for each state, caching to a JSONL file."""
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cached: dict[str, dict[str, Any]] = {}
    if cache_path.exists():
        with cache_path.open() as f:
            for line in f:
                line = line.strip()
                if line:
                    r = json.loads(line)
                    cached[r['state']] = r

    results: list[dict[str, Any]] = []
    with cache_path.open('a') as f:
        for s in states:
            if s in cached:
                results.append(cached[s])
                continue
            logger.info(f'Computing {s} -> {cache_path.name}')
            r = compute_fn(s, state_geoms[s])
            r = {'state': s, **r}
            f.write(json.dumps(r) + '\n'); f.flush()
            results.append(r)
    results.sort(key=lambda r: r['state'])
    return results

## 1. CONUS confusion matrix: CDL 2020 row crop × GLAD GLC 2020 cropland

We compute the 2×2 confusion matrix per CONUS state, then sum to CONUS totals. **CDL row crop** = `CDL_ROWCROP_CODES` above (excludes hay and fallow). **GLAD GLC cropland** = pixel value 244 in the LCLUC v2 maps.

In [ ]:
# Confusion matrix encoding: cdl_rc * 2 + glc_crop -> 0..3
COMBO_KEYS_2x2 = {
    0: 'nonRC_glc_noncrop',
    1: 'nonRC_glc_crop',
    2: 'RC_glc_noncrop',
    3: 'RC_glc_crop',
}


def confusion_2x2_for_geom(geometry: EEGeometry) -> dict[str, float]:
    cdl_mask = load_cdl_rowcrop_mask(CDL_YEAR, geometry)
    glad_mask = load_glad_glc_crop_mask(GLAD_YEAR, geometry)
    pixel_area_ha = ee.Image.pixelArea().divide(10000)
    combo = cdl_mask.multiply(2).add(glad_mask).rename('combo')
    result = pixel_area_ha.addBands(combo).reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='combo'),
        geometry=geometry, scale=30, maxPixels=1e13, bestEffort=True,
    )
    groups = ee.List(result.get('groups')).getInfo()
    areas = {v: 0.0 for v in COMBO_KEYS_2x2.values()}
    for g in groups or []:
        code = int(g['combo'])
        if code in COMBO_KEYS_2x2:
            areas[COMBO_KEYS_2x2[code]] = float(g['sum'])
    return areas


def compute_state_confusion(state: str, geometry: EEGeometry) -> dict[str, float]:
    if state in LARGE_STATES:
        totals = {v: 0.0 for v in COMBO_KEYS_2x2.values()}
        for tile in tile_geometry(geometry):
            for k, v in confusion_2x2_for_geom(tile).items():
                totals[k] += v
        return totals
    return confusion_2x2_for_geom(ee.Geometry(geometry))


state_geoms = get_conus_states()
all_states = sorted(state_geoms.keys())

confusion_results = cached_per_state(
    OUTPUT_DIR / 'cdl_rowcrop_glc_confusion.jsonl',
    all_states, state_geoms, compute_state_confusion,
)

df_conf = pd.DataFrame(confusion_results).set_index('state')
df_conf['rc_total'] = df_conf['RC_glc_crop'] + df_conf['RC_glc_noncrop']
df_conf['rc_confirm_pct'] = 100 * df_conf['RC_glc_crop'] / df_conf['rc_total']

conus = df_conf[list(COMBO_KEYS_2x2.values())].sum()
rc_total = conus['RC_glc_crop'] + conus['RC_glc_noncrop']
ncrop_total = conus['nonRC_glc_crop'] + conus['nonRC_glc_noncrop']
glc_crop_total = conus['RC_glc_crop'] + conus['nonRC_glc_crop']

summary = pd.DataFrame(
    [[conus['RC_glc_crop'], conus['RC_glc_noncrop'], rc_total],
     [conus['nonRC_glc_crop'], conus['nonRC_glc_noncrop'], ncrop_total],
     [glc_crop_total, conus['RC_glc_noncrop'] + conus['nonRC_glc_noncrop'],
      rc_total + ncrop_total]],
    columns=['GLAD GLC cropland', 'GLAD GLC non-cropland', 'Total'],
    index=['CDL row crop', 'CDL not row crop', 'Total'],
)
print('CONUS CDL 2020 row crop × GLAD GLC 2020 cropland (hectares):')
print(summary.applymap(lambda x: f'{x:>15,.0f}').to_string())
print(f'\nGLC confirmation rate for CDL row crops: '
      f'{100 * conus["RC_glc_crop"] / rc_total:.1f}%')
print(f'CDL row crops not confirmed by GLC: '
      f'{conus["RC_glc_noncrop"]:,.0f} ha '
      f'({100 * conus["RC_glc_noncrop"] / rc_total:.1f}% of CDL row crop)')

## 2. Confirmation rate by region

Disagreement is not uniform across the country. The Corn Belt and Great Plains have very high agreement (large fields are easy to classify); the Southeast and Northeast have substantially lower agreement (smaller, more fragmented fields).

In [ ]:
REGIONS = {
    'Corn Belt (IA, IL, IN, NE, OH)': ['Iowa', 'Illinois', 'Indiana', 'Nebraska', 'Ohio'],
    'Great Plains (ND, SD, KS, MT)': ['North Dakota', 'South Dakota', 'Kansas', 'Montana'],
    'Southeast (AL, FL, GA, SC)': ['Alabama', 'Florida', 'Georgia', 'South Carolina'],
    'Northeast (CT, MA, RI, PA)': ['Connecticut', 'Massachusetts', 'Rhode Island', 'Pennsylvania'],
}

rows = []
for region, members in REGIONS.items():
    sub = df_conf.loc[df_conf.index.intersection(members)]
    rc = sub['RC_glc_crop'].sum() + sub['RC_glc_noncrop'].sum()
    confirm_pct = 100 * sub['RC_glc_crop'].sum() / rc if rc else 0
    state_pcts = sub['rc_confirm_pct'].sort_values()
    rows.append({
        'region': region,
        'rc_total_ha': int(rc),
        'confirm_pct_aggregate': f'{confirm_pct:.1f}%',
        'per_state_range': f'{state_pcts.min():.0f}–{state_pcts.max():.0f}%',
    })

print(pd.DataFrame(rows).to_string(index=False))

## 3. Decomposing the lost CDL row crop pixels

To understand the emissions impact of the 9% disagreement pixels, we decompose the lost pixels into three cases on a 10-state agricultural sample (representative of the bulk of US row crop production):

- **Case A** — GLAD GLC 2020 = cropland. Standard agreement; included in the methodology.
- **Case B** — GLAD GLC 2020 ≠ cropland, but GLC class changed 2000→2020. GLC says a transition occurred but to a non-crop destination.
- **Case C** — GLAD GLC 2020 ≠ cropland and GLC class is stable 2000→2020. CDL says row crop, GLC says it has never been crop.

Case A is the everyday agreement. Case B and Case C together are the disagreement. Case C is the larger of the two, but a big chunk of it is built-up infrastructure that the methodology would not have generated emissions for anyway.

In [ ]:
def case_abc_for_geom(geometry: EEGeometry) -> dict[str, Any]:
    cdl_rc = load_cdl_rowcrop_mask(CDL_YEAR, geometry)
    glc_2000_cat = classify_glad_glc(
        ee.Image(GEE_GLAD_GLC_PREFIX + '2000').clip(geometry)
    ).rename('glc2000')
    glc_2020_cat = classify_glad_glc(
        ee.Image(GEE_GLAD_GLC_PREFIX + '2020').clip(geometry)
    ).rename('glc2020')
    glc_2020_is_crop = glc_2020_cat.eq(6)
    glc_changed = glc_2000_cat.neq(glc_2020_cat)
    pixel_area_ha = ee.Image.pixelArea().divide(10000)
    masked = pixel_area_ha.updateMask(cdl_rc)

    def _sum(mask: EEImage) -> float:
        r = masked.updateMask(mask).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=geometry,
            scale=30, maxPixels=1e13, bestEffort=True,
        ).getInfo()
        return float(r.get('area', 0) or 0)

    case_a = _sum(glc_2020_is_crop)
    case_b_mask = glc_2020_is_crop.Not().And(glc_changed)
    case_c_mask = glc_2020_is_crop.Not().And(glc_changed.Not())
    case_b = _sum(case_b_mask)
    case_c = _sum(case_c_mask)

    # Case B by GLC 2000 source x GLC 2020 destination (encoded src*10 + dst)
    combo = glc_2000_cat.multiply(10).add(glc_2020_cat).rename('combo')
    b_groups = (
        masked.updateMask(case_b_mask)
        .addBands(combo.updateMask(case_b_mask))
        .reduceRegion(
            reducer=ee.Reducer.sum().group(groupField=1, groupName='combo'),
            geometry=geometry, scale=30, maxPixels=1e13, bestEffort=True,
        )
    )
    case_b_breakdown: dict[str, float] = {}
    for g in ee.List(b_groups.get('groups')).getInfo() or []:
        c = int(g['combo'])
        src = GLAD_CATEGORY_NAMES.get(c // 10, f'unk_{c // 10}')
        dst = GLAD_CATEGORY_NAMES.get(c % 10, f'unk_{c % 10}')
        case_b_breakdown[f'{src} -> {dst}'] = float(g['sum'])

    # Case C by GLC 2020 stable land cover
    c_groups = (
        masked.updateMask(case_c_mask)
        .addBands(glc_2020_cat.updateMask(case_c_mask))
        .reduceRegion(
            reducer=ee.Reducer.sum().group(groupField=1, groupName='glc'),
            geometry=geometry, scale=30, maxPixels=1e13, bestEffort=True,
        )
    )
    case_c_breakdown: dict[str, float] = {}
    for g in ee.List(c_groups.get('groups')).getInfo() or []:
        name = GLAD_CATEGORY_NAMES.get(int(g['glc']), f"unk_{int(g['glc'])}")
        case_c_breakdown[name] = float(g['sum'])

    return {
        'case_a': case_a, 'case_b': case_b, 'case_c': case_c,
        'case_b_breakdown': case_b_breakdown,
        'case_c_breakdown': case_c_breakdown,
    }


def compute_state_abc(state: str, geometry: EEGeometry) -> dict[str, Any]:
    if state in LARGE_STATES:
        agg = {'case_a': 0.0, 'case_b': 0.0, 'case_c': 0.0,
               'case_b_breakdown': {}, 'case_c_breakdown': {}}
        for tile in tile_geometry(geometry):
            r = case_abc_for_geom(tile)
            for k in ('case_a', 'case_b', 'case_c'):
                agg[k] += r[k]
            for k, v in r['case_b_breakdown'].items():
                agg['case_b_breakdown'][k] = agg['case_b_breakdown'].get(k, 0.0) + v
            for k, v in r['case_c_breakdown'].items():
                agg['case_c_breakdown'][k] = agg['case_c_breakdown'].get(k, 0.0) + v
        return agg
    return case_abc_for_geom(ee.Geometry(geometry))


abc_results = cached_per_state(
    OUTPUT_DIR / 'cdl_rowcrop_glc_case_abc.jsonl',
    STATES_FOR_DEEP_DIVE, state_geoms, compute_state_abc,
)

df_abc = pd.DataFrame([
    {'state': r['state'], 'Case A (ha)': r['case_a'],
     'Case B (ha)': r['case_b'], 'Case C (ha)': r['case_c']}
    for r in abc_results
])
df_abc['Total (ha)'] = df_abc[['Case A (ha)', 'Case B (ha)', 'Case C (ha)']].sum(axis=1)
for c in ('A', 'B', 'C'):
    df_abc[f'{c}%'] = (100 * df_abc[f'Case {c} (ha)'] / df_abc['Total (ha)']).round(1)

totals = {
    'state': '10-STATE TOTAL',
    'Case A (ha)': df_abc['Case A (ha)'].sum(),
    'Case B (ha)': df_abc['Case B (ha)'].sum(),
    'Case C (ha)': df_abc['Case C (ha)'].sum(),
}
totals['Total (ha)'] = totals['Case A (ha)'] + totals['Case B (ha)'] + totals['Case C (ha)']
for c in ('A', 'B', 'C'):
    totals[f'{c}%'] = round(100 * totals[f'Case {c} (ha)'] / totals['Total (ha)'], 1)
df_abc = pd.concat([df_abc, pd.DataFrame([totals])], ignore_index=True)

print(df_abc.to_string(index=False, formatters={
    'Case A (ha)': lambda x: f'{x:>14,.0f}',
    'Case B (ha)': lambda x: f'{x:>14,.0f}',
    'Case C (ha)': lambda x: f'{x:>14,.0f}',
    'Total (ha)':  lambda x: f'{x:>14,.0f}',
}))

### 3a. Case B: source → destination breakdown

For Case B pixels, GLAD GLC saw a transition between 2000 and 2020 — but the destination was not classified as cropland. We aggregate the 10-state Case B totals by GLC 2000 source and GLC 2020 destination class. The dominant category is pixels GLC saw as cropland in 2000 and reclassified to non-cropland by 2020.

Genuine emissions-relevant transitions (forest → non-crop, short vegetation → non-crop, wetland → non-crop) sum to a small fraction of total disagreement.

In [ ]:
case_b_combined: dict[str, float] = {}
for r in abc_results:
    for k, v in r['case_b_breakdown'].items():
        case_b_combined[k] = case_b_combined.get(k, 0.0) + v

total_disagreement = sum(r['case_b'] + r['case_c'] for r in abc_results)

# Aggregate to source-only buckets matching Appendix 1 framing
# (destination is non-cropland by construction, so we summarize by source)
SOURCE_BUCKETS = {
    'cropland': 'Cropland',
    'short_veg': 'Short veg',
    'forest': 'Forest',
    'wetland_short_veg': 'Wetland',
    'wetland_forest': 'Wetland',
}
by_source: dict[str, float] = {}
for transition, area in case_b_combined.items():
    src = transition.split(' -> ')[0]
    label = SOURCE_BUCKETS.get(src, 'Water/bare/other')
    by_source[label] = by_source.get(label, 0.0) + area

rows = []
for label in ('Cropland', 'Short veg', 'Forest', 'Wetland', 'Water/bare/other'):
    area = by_source.get(label, 0.0)
    rows.append({
        'GLC 2000 source -> GLC 2020 destination (non-cropland)': f'{label} -> non-cropland',
        'Area (ha)': int(area),
        '% of disagreement': f'{100 * area / total_disagreement:.1f}%',
    })
print(pd.DataFrame(rows).to_string(index=False))

print('\nTop individual src->dst transitions in Case B:')
for k in sorted(case_b_combined, key=case_b_combined.get, reverse=True)[:10]:
    print(f'  {k:<40} {case_b_combined[k]:>14,.0f} ha')

### 3b. Case C: stable GLC land cover breakdown

For Case C pixels, GLC sees the pixel as the same non-cropland class in 2000 and 2020. CDL says row crop; GLC says it has been something other than cropland for the whole study period. The dominant component (~44%) is **stable built-up** — small structures, farmsteads, grain bins, paved infrastructure. The next biggest is **stable short vegetation** (~30%) — perhaps grassed field margins, hayfields with crop edges, and CRP land that CDL is calling row crop too aggressively.

In [ ]:
case_c_combined: dict[str, float] = {}
for r in abc_results:
    for k, v in r['case_c_breakdown'].items():
        case_c_combined[k] = case_c_combined.get(k, 0.0) + v

DEST_BUCKETS = {
    'built_up': 'Built-up',
    'short_veg': 'Short vegetation',
    'forest': 'Forest',
    'wetland_short_veg': 'Wetland',
    'wetland_forest': 'Wetland',
}
by_dest: dict[str, float] = {}
for cls, area in case_c_combined.items():
    label = DEST_BUCKETS.get(cls, 'Water/bare/other')
    by_dest[label] = by_dest.get(label, 0.0) + area

rows = []
for label in ('Built-up', 'Short vegetation', 'Forest', 'Wetland', 'Water/bare/other'):
    area = by_dest.get(label, 0.0)
    rows.append({
        'GLC 2020 class (stable since 2000)': label,
        'Area (ha)': int(area),
        '% of disagreement': f'{100 * area / total_disagreement:.1f}%',
    })
print(pd.DataFrame(rows).to_string(index=False))

## 4. Why row crops only (not the full CDL crop class)

CDL distinguishes row crops, hay/alfalfa, and fallow/idle within its broader "crop" universe. An earlier version of this analysis used the full CDL crop class and saw very poor agreement with GLAD over time:

- **Row crops:** ~95% confirmed by GLAD
- **Hay/alfalfa:** only ~56% confirmed
- **Fallow/idle:** ~83% confirmed but flickers heavily year-to-year

Over 2011–2019, CDL reported 11.3 M ha of net cropland expansion, but GLAD only 4.2 M ha — a 2.7× gap. Decomposing CDL's expansion shows that **75% of the difference comes from hay/fallow churn**, not row crop change:

| CDL category | 2011 (ha)   | 2019 (ha)   | Net (ha)   | % of CDL expansion |
|---|---:|---:|---:|---:|
| Row crop     | 102,951,573 | 105,820,139 | +2,868,566 | 25.5% |
| Hay/alfalfa  |  16,512,494 |  21,517,515 | +5,005,021 | 44.5% |
| Fallow       |  10,599,070 |  13,976,681 | +3,377,611 | 30.0% |

Row crop expansion alone is reasonably close to GLAD's signal.

The full code to reproduce this analysis (including state-level breakdown of hay reclassification in the Southeast and ranching states, and fallow flicker in the Great Plains) is preserved below for completeness; results are cached to `output/cdl_glad_expansion.jsonl`.

In [ ]:
from jdluc.utils.constants import GEE_GLAD_CROPLAND_PREFIX

EXPANSION_START_YEAR = 2011
EXPANSION_END_YEAR = 2019

# 8-cell encoding: cdl_category * 2 + glad_binary
EXPANSION_KEYS = {
    0: 'noncrop_glad_noncrop', 1: 'noncrop_glad_crop',
    2: 'rowcrop_glad_noncrop', 3: 'rowcrop_glad_crop',
    4: 'hay_glad_noncrop', 5: 'hay_glad_crop',
    6: 'fallow_glad_noncrop', 7: 'fallow_glad_crop',
}


def load_cdl_4cat(year: int, geometry: EEGeometry) -> EEImage:
    """CDL as 0=non-crop, 1=row crop, 2=hay, 3=fallow."""
    cdl = (
        ee.ImageCollection(GEE_CDL_COLLECTION)
        .filter(ee.Filter.calendarRange(year, year, 'year'))
        .first().select('cropland').clip(geometry)
    )
    codes = CDL_ROWCROP_CODES + CDL_HAY_CODES + CDL_FALLOW_CODES
    vals = ([1] * len(CDL_ROWCROP_CODES)
            + [2] * len(CDL_HAY_CODES)
            + [3] * len(CDL_FALLOW_CODES))
    return cdl.remap(codes, vals, defaultValue=0).rename('cdl_cat')


def load_glad_binary_crop_mask(year: int, geometry: EEGeometry) -> EEImage:
    """GLAD binary cropland mask (Potapov 30m). Different asset from GLC LCLUC."""
    return (
        ee.ImageCollection(GEE_GLAD_CROPLAND_PREFIX + str(year))
        .mosaic().clip(geometry).eq(1).rename('glad_crop')
    )


def expansion_confusion_for_geom(geometry: EEGeometry, year: int) -> dict[str, float]:
    cdl_cat = load_cdl_4cat(year, geometry)
    glad = load_glad_binary_crop_mask(year, geometry)
    pixel_area_ha = ee.Image.pixelArea().divide(10000)
    combo = cdl_cat.multiply(2).add(glad).rename('combo')
    result = pixel_area_ha.addBands(combo).reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='combo'),
        geometry=geometry, scale=30, maxPixels=1e13, bestEffort=True,
    )
    areas = {v: 0.0 for v in EXPANSION_KEYS.values()}
    for g in ee.List(result.get('groups')).getInfo() or []:
        code = int(g['combo'])
        if code in EXPANSION_KEYS:
            areas[EXPANSION_KEYS[code]] = float(g['sum'])
    return areas


def compute_state_expansion(state: str, geometry: EEGeometry) -> dict[str, Any]:
    def _both_years(geom):
        return {
            'start': expansion_confusion_for_geom(geom, EXPANSION_START_YEAR),
            'end':   expansion_confusion_for_geom(geom, EXPANSION_END_YEAR),
        }
    if state in LARGE_STATES:
        agg_s = {v: 0.0 for v in EXPANSION_KEYS.values()}
        agg_e = dict(agg_s)
        for tile in tile_geometry(geometry):
            r = _both_years(tile)
            for k, v in r['start'].items(): agg_s[k] += v
            for k, v in r['end'].items():   agg_e[k] += v
        return {'start': agg_s, 'end': agg_e}
    return _both_years(ee.Geometry(geometry))


# Uncomment to run the historical expansion analysis (~1-2 hours of GEE).
# expansion_results = cached_per_state(
#     OUTPUT_DIR / 'cdl_glad_expansion.jsonl',
#     all_states, state_geoms, compute_state_expansion,
# )